<a href="https://colab.research.google.com/github/Vestal1/nlp-analyse-klarschiff-hro/blob/main/notebooks/klarschiff_nlp_analyse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Untersuchung der Klarschiff.HRO-Meldungen

#In diesem Projekt wird der Datensatz Klarschiff.HRO mit einfachen Verfahren aus dem Bereich Natural Language Processing untersucht. Zuerst wird der Datensatz betrachtet und bereinigt. Danach werden CountVectorizer und TF-IDF sowie die Verfahren LDA und NMF ausprobiert.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Bibliotheken wurden geladen.")



Bibliotheken wurden geladen.


## 1. Datensatz einlesen

#Zuerst wird der Datensatz direkt von der Open-Data-Seite der Stadt Rostock eingelesen. Danach werden die ersten Zeilen und die Größe des Datensatzes betrachtet.

In [3]:
url = "https://geo.sv.rostock.de/download/opendata/klarschiffhro-meldungen/klarschiffhro-meldungen.csv"

df = pd.read_csv(url)

print("Anzahl Zeilen und Spalten:", df.shape)
df.head()

Anzahl Zeilen und Spalten: (2619, 13)


,latitude,longitude,nummer,typ,hauptkategorie,unterkategorie,status,statusinformation,unterstuetzungen,beschreibung,erstellungsdatum,letztes_aenderungsdatum,aktuelle_zustaendigkeit
0,54.125287,12.141777,124715,Problem,Müll/Schmutz,Sperrmüll,in Bearbeitung,Die Meldung wurde an die Stadtentsorgung Rosto...,NaN,Möbelteile,2026/09/04,2026/09/08,Amt für Umwelt- und Klimaschutz [delegiert an:...
1,54.139035,12.047776,124377,Problem,Müll/Schmutz,Altfahrzeug/Schrottfahrrad,in Bearbeitung,Vielen Dank für Ihren Hinweis. Der Prozess zur...,NaN,Erstaufnahme abgestelltes Fahrzeug Ausländisch...,2026/08/28,2026/09/07,Amt für Umwelt- und Klimaschutz
2,54.122286,12.053965,123263,Problem,Müll/Schmutz,Sperrmüll,offen,NaN,NaN,NaN,2026/08/03,2026/08/03,Amt für Umwelt- und Klimaschutz
3,54.083808,12.115074,124576,Problem,Müll/Schmutz,Elektroschrott,gelöst,Die Meldung wurde an die Stadtentsorgung Rosto...,NaN,Gegenüber der Hundertmännerstr. 4 befindet sic...,2026/09/01,2026/09/08,Amt für Umwelt- und Klimaschutz [delegiert an:...
4,54.080055,12.185793,122967,Problem,Müll/Schmutz,Sperrmüll,offen,NaN,NaN,NaN,2026/07/25,2026/07/25,Amt für Umwelt- und Klimaschutz


In [4]:
print("Spaltennamen:")
print(df.columns.tolist())

print("\nFehlende Werte:")
print(df.isna().sum().sort_values(ascending=False))

Spaltennamen:
['latitude', 'longitude', 'nummer', 'typ', 'hauptkategorie', 'unterkategorie', 'status', 'statusinformation', 'unterstuetzungen', 'beschreibung', 'erstellungsdatum', 'letztes_aenderungsdatum', 'aktuelle_zustaendigkeit']

Fehlende Werte:
unterstuetzungen           2037
statusinformation           996
beschreibung                364
longitude                     0
latitude                      0
hauptkategorie                0
typ                           0
nummer                        0
status                        0
unterkategorie                0
erstellungsdatum              0
letztes_aenderungsdatum       0
aktuelle_zustaendigkeit       0
dtype: int64


## 2. Untersuchung der Beschreibungstexte

Für die weitere Analyse wird hauptsächlich die Spalte `beschreibung` benötigt. Da einige Meldungen keine Beschreibung enthalten, werden diese später entfernt. Vorher wird geprüft, wie viele Texte vorhanden sind und ob sehr kurze oder doppelte Beschreibungen vorkommen.

In [5]:
texte = df["beschreibung"].dropna().astype(str).str.strip()
texte = texte[texte != ""]

wortanzahl = texte.str.split().str.len()

print("Vorhandene Beschreibungstexte:", len(texte))
print("Eindeutige Beschreibungstexte:", texte.nunique())
print("Exakte Wiederholungen:", texte.duplicated().sum())
print("Mittlere Wortanzahl:", round(wortanzahl.mean(), 1))
print("Texte mit weniger als 5 Wörtern:", (wortanzahl < 5).sum())

Vorhandene Beschreibungstexte: 2255
Eindeutige Beschreibungstexte: 2088
Exakte Wiederholungen: 167
Mittlere Wortanzahl: 26.5
Texte mit weniger als 5 Wörtern: 431


## 3. Auswahl der Texte

Meldungen ohne Beschreibung können für die Textanalyse nicht verwendet werden. Exakte Wiederholungen werden entfernt, damit gleiche Texte die Ergebnisse nicht zu stark beeinflussen. Außerdem werden Beschreibungen mit weniger als fünf Wörtern ausgeschlossen, da diese meistens nur wenig Inhalt bieten.

In [6]:
df_text = df.dropna(subset=["beschreibung"]).copy()

df_text["beschreibung"] = (
    df_text["beschreibung"]
    .astype(str)
    .str.strip()
)

df_text = df_text[df_text["beschreibung"] != ""]

anzahl_vorher = len(df_text)

df_text = df_text.drop_duplicates(subset=["beschreibung"])

duplikate_entfernt = anzahl_vorher - len(df_text)

df_text["wortanzahl"] = df_text["beschreibung"].str.split().str.len()

kurze_texte = (df_text["wortanzahl"] < 5).sum()

df_text = df_text[df_text["wortanzahl"] >= 5].copy()

print("Ausgangstexte:", anzahl_vorher)
print("Duplikate entfernt:", duplikate_entfernt)
print("Kurze Texte entfernt:", kurze_texte)
print("Verbleibende Texte:", len(df_text))

Ausgangstexte: 2255
Duplikate entfernt: 167
Kurze Texte entfernt: 281
Verbleibende Texte: 1807


Datensatz geprüft und Texte ausgewählt